# Notebook 2: RAG Triad Diagnostics — Retrieval vs Generation (**Ragas**)

When a RAG chatbot gives the wrong answer, you must isolate **where** it failed before changing code:
- **Search / Retrieval failure:** Did the retriever miss the right policy document? (**`Context Recall`** / **`Context Precision`**)
- **LLM / Generation failure:** Did the LLM receive the right document yet invent false claims? (**`Faithfulness`**)

Let's diagnose both failure modes cleanly using **`Ragas`**.

## Step 1: Run 3 Scenarios and Observe the Difference
1. **Healthy RAG**
2. **Broken Search (`retriever_bug=True`)** — retriever fetches the Shipping doc instead of the Return doc!
3. **Lying LLM (`bug_mode='hallucination'`)** — retriever fetches `DOC-RETURN`, yet the LLM contradicts it!

In [ ]:
import sys
import pandas as pd

if ".." not in sys.path:
    sys.path.append("..")

from lankashop import POLICY_DOCS, ask_agent, reset_db, run_async, set_live_mode

USE_LIVE_API = False  # Set True to score Faithfulness via OpenAI
set_live_mode(USE_LIVE_API)

query = "Can I return Wireless Earbuds if I already opened the box?"
scenarios = [
    ("1. Healthy RAG", "none", False),
    ("2. Broken Search (retriever_bug=True)", "none", True),
    ("3. Lying LLM (hallucination)", "hallucination", False),
]

traces = {}
for name, bm, rb in scenarios:
    reset_db()
    traces[name] = ask_agent(query, bug_mode=bm, retriever_bug=rb)
    print(f"{name} -> Docs: {traces[name]['retrieved_doc_ids']} | Answer: {traces[name]['answer']}")

## Step 2: Diagnose Root Cause with `Ragas` (`Context Recall` & `Faithfulness`)

In [ ]:
import os
import ragas
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics._context_recall import NonLLMContextRecall

print("Ragas version:", ragas.__version__)
recall_metric = NonLLMContextRecall()
RETURN_DOC = POLICY_DOCS[0]["text"]

def score_faithfulness(user_query: str, contexts: list[str], answer: str) -> float:
    if USE_LIVE_API and os.getenv("OPENAI_API_KEY", "").strip():
        from langchain_openai import ChatOpenAI
        from ragas.llms import LangchainLLMWrapper
        from ragas.metrics._faithfulness import Faithfulness

        judge = LangchainLLMWrapper(ChatOpenAI(model=os.getenv("OPENAI_MODEL", "gpt-4o-mini"), temperature=0))
        sample = SingleTurnSample(user_input=user_query, response=answer, retrieved_contexts=contexts)
        return float(run_async(Faithfulness(llm=judge).single_turn_ascore(sample)))

    # Lightweight $0 offline faithfulness check
    ctx_text, ans_text = " ".join(contexts).lower(), answer.lower()
    if "cannot be returned" in ctx_text and "can always be returned" in ans_text:
        return 0.0  # Direct contradiction (hallucination)
    if "shipping policy" in ctx_text and "return" not in ctx_text:
        return 0.5  # Grounded in the wrong retrieved document
    return 1.0

results = []
for name, _, _ in scenarios:
    tr = traces[name]
    sample = SingleTurnSample(
        user_input=query,
        response=tr["answer"],
        retrieved_contexts=tr["retrieved_contexts"],
        reference_contexts=[RETURN_DOC],
    )
    recall = float(run_async(recall_metric.single_turn_ascore(sample)))
    faith = score_faithfulness(query, tr["retrieved_contexts"], tr["answer"])
    diag = "FIX SEARCH (Retrieval Miss)" if recall < 0.7 else ("FIX PROMPT (Hallucination)" if faith < 0.8 else "PASS")
    results.append({"scenario": name, "context_recall": round(recall, 2), "faithfulness": round(faith, 2), "diagnosis": diag})

pd.DataFrame(results)

## Step 3: Intuition Behind `Context Precision` (Signal vs Noise Rank)
When multiple chunks are retrieved, **`Context Precision`** measures whether the relevant document was ranked near the top rather than buried behind noisy chunks.

In [ ]:
def simple_context_precision(retrieved_doc_ids: list[str], target_doc_id: str) -> float:
    if target_doc_id not in retrieved_doc_ids:
        return 0.0
    rank = retrieved_doc_ids.index(target_doc_id) + 1
    return round(1.0 / rank, 2)

print("Clean Retrieval [DOC-RETURN]:", simple_context_precision(["DOC-RETURN"], "DOC-RETURN"))
print("Noisy Retrieval [DOC-SHIPPING, DOC-RETURN]:", simple_context_precision(["DOC-SHIPPING", "DOC-RETURN"], "DOC-RETURN"))
print("Missing Doc [DOC-SHIPPING]:", simple_context_precision(["DOC-SHIPPING"], "DOC-RETURN"))